# Classicmodels customer API paths (`main.py`)

This notebook calls the **`/customers`** routes from **`app/main.py`** over HTTP using the **`requests`** package.

**Prerequisite:** Run the API from the repository root, for example **`uvicorn app.main:app --reload --port 8000`**. The default base URL is **`http://127.0.0.1:8000`**; override with the environment variable **`API_BASE_URL`** if you use another host or port.

**Note:** Cells that **`POST`**, **`PUT`**, or **`DELETE`** modify the **`classicmodels`** database. Run create/update/delete cells once, or re-run only after the created row has been deleted.

In [ ]:
import os

import requests

BASE_URL = os.environ.get("API_BASE_URL", "http://127.0.0.1:8000").rstrip("/")

try:
    _health = requests.get(f"{BASE_URL}/health", timeout=5)
    _health.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        f"Cannot reach API at {BASE_URL!r}. From the repo root run e.g. "
        "`uvicorn app.main:app --reload --port 8000`, then rerun this cell "
        "(or set API_BASE_URL if the server uses another host/port)."
    ) from exc

# customerNumber 103 is 'Atelier graphique' — present in the standard classicmodels dataset.
SAMPLE_CUSTOMER_NUMBER = 103

## `GET /customers`

List/search with optional query parameters: `customerName`, `city`, `country` (equality template).

In [ ]:
resp = requests.get(f"{BASE_URL}/customers", timeout=30)
assert resp.status_code == 200, resp.text
data = resp.json()
assert "items" in data
print("count:", len(data["items"]))
data["items"][:3]

In [ ]:
resp = requests.get(
    f"{BASE_URL}/customers", params={"country": "France"}, timeout=30
)
assert resp.status_code == 200
items = resp.json()["items"]
assert all(r["country"] == "France" for r in items)
len(items)

## `GET /customers/{customerNumber}`

Returns **`404`** if the customer does not exist.

In [ ]:
resp = requests.get(f"{BASE_URL}/customers/{SAMPLE_CUSTOMER_NUMBER}", timeout=30)
assert resp.status_code == 200
resp.json()

In [ ]:
missing = requests.get(f"{BASE_URL}/customers/999999", timeout=30)
assert missing.status_code == 404
missing.json()

## `POST /customers`

Creates a customer. Response body is the new `customerNumber`.

In [ ]:
payload = {
    "customerNumber": 50000,
    "customerName": "Notebook Test Customer",
    "contactLastName": "Test",
    "contactFirstName": "Notebook",
    "phone": "555-0100",
    "addressLine1": "1 Test Street",
    "city": "New York",
    "country": "USA",
}
resp = requests.post(f"{BASE_URL}/customers", json=payload, timeout=30)
assert resp.status_code == 200, resp.text
new_id = resp.json()
print("created customerNumber:", new_id)
new_id

## `PUT /customers/{customerNumber}`

Updates by `customerNumber`; **`400`** if the customer does not exist.

In [ ]:
# Uses `new_id` from the POST cell above — run that cell first.
update_body = {
    "customerNumber": new_id,
    "customerName": "Notebook Test Customer (updated)",
    "contactLastName": "Test",
    "contactFirstName": "Notebook",
    "phone": "555-0199",
    "addressLine1": "1 Test Street",
    "city": "Boston",
    "country": "USA",
}
resp = requests.put(f"{BASE_URL}/customers/{new_id}", json=update_body, timeout=30)
assert resp.status_code == 200, resp.text
assert resp.json() == {"updated": 1}
requests.get(f"{BASE_URL}/customers/{new_id}", timeout=30).json()

## `DELETE /customers/{customerNumber}`

Returns **`{"deleted": 0}`** or **`{"deleted": 1}`**.

In [ ]:
resp = requests.delete(f"{BASE_URL}/customers/{new_id}", timeout=30)
assert resp.status_code == 200
assert resp.json()["deleted"] == 1

gone = requests.get(f"{BASE_URL}/customers/{new_id}", timeout=30)
assert gone.status_code == 404
gone.json()